In [1]:
import os
import numpy as np
import pandas as pd
from datetime import date
import ClientProfile
import MarketData
import Portfolio
import ResearchViews
import RiskEngine
import Reporter
import TradeBlotter

In [2]:
def _print_section(title: str) -> None:
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

In [3]:
# Initialization cell
def initialize_framework():
    global portfolio, market_data, client_profile, research_views
    
    # Initialize core components
    portfolio = Portfolio.Portfolio()
    market_data = MarketData.MarketData()
    client_profile = ClientProfile.ClientProfile()
    research_views = ResearchViews.ResearchViews()

In [4]:
initialize_framework()
universe = market_data.build_universe(period="1y")
returns = market_data.universe_returns()
    
sample_ticker = "AAPL"
stock_data = market_data.get_stock(sample_ticker, start="2020-01-01", end="2020-12-31")

[*********************100%***********************]  1 of 1 completed


In [5]:
returns

,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2025-03-31,0.002485,0.019413,0.020605,-0.010191,0.013989,0.011782,0.025334,-0.005652,0.000546,0.002925,...,-0.002725,0.014211,0.015347,0.010193,0.000167,-0.018073,0.010272,0.010355,0.006304,0.009441
2025-04-01,-0.022976,0.004772,-0.015512,0.014398,-0.007237,-0.000416,0.004935,-0.000860,-0.012347,0.004166,...,-0.004781,-0.009221,-0.000989,0.000925,0.003516,0.019878,0.009914,-0.007157,0.001805,-0.016641
2025-04-02,0.017010,0.003136,-0.005236,0.013369,0.001822,0.007073,0.008770,0.006733,0.002058,-0.004978,...,0.011668,-0.014384,0.002545,-0.003108,-0.006923,0.044216,-0.004342,0.001157,0.015897,0.009820
2025-04-03,-0.054487,-0.092456,-0.017301,-0.071906,-0.002274,-0.005577,-0.047008,-0.048033,-0.093742,-0.008339,...,-0.069539,-0.106206,0.017489,-0.052583,-0.067703,-0.067231,0.020350,0.001422,-0.170637,-0.023792
2025-04-04,-0.060819,-0.072887,-0.072803,-0.064140,-0.054623,-0.087765,-0.054402,-0.049503,-0.090004,-0.089342,...,-0.042654,-0.038282,-0.058913,-0.071956,-0.057573,-0.068742,-0.084361,-0.047222,-0.062935,-0.049370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-23,0.006469,0.014113,-0.000683,0.031668,-0.005784,0.014735,0.000150,-0.002055,0.008920,0.027505,...,0.030708,0.011502,0.002345,0.009144,-0.004151,0.030318,0.013618,-0.000797,0.011050,-0.003804
2026-03-24,0.019461,0.000596,0.001318,-0.019534,-0.007535,-0.000641,-0.032397,-0.035414,0.030879,0.050743,...,0.016408,0.016217,0.013125,0.026376,0.004252,-0.020925,0.005614,-0.016857,0.007869,0.006075
2026-03-25,-0.010683,0.003894,0.009649,0.013923,0.007400,-0.002991,-0.006459,-0.006782,0.000621,0.003080,...,0.004673,-0.008563,-0.003335,-0.012819,0.010793,0.002004,-0.001693,0.015292,0.004627,0.006728


In [6]:
stock_data

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,,
2020-01-02,72.333878,75.087502,75.150002,73.797501,74.059998,135480400
2020-01-03,71.630638,74.357498,75.144997,74.125000,74.287498,146322800
2020-01-06,72.201416,74.949997,74.989998,73.187500,73.447502,118387200
2020-01-07,71.861847,74.597504,75.224998,74.370003,74.959999,108872000
2020-01-08,73.017838,75.797501,76.110001,74.290001,74.290001,132079200
...,...,...,...,...,...,...
2020-12-23,127.246902,130.960007,132.429993,130.779999,132.160004,88223700
2020-12-24,128.228271,131.970001,133.460007,131.100006,131.320007,54930100


In [7]:
def calculate_covariance_matrix(X):
    if isinstance(X, pd.DataFrame):
        data = X.dropna(how="any")
        if data.empty:
            return pd.DataFrame()
        covariance_matrix = data.cov()
        return covariance_matrix

    data = np.asarray(X, dtype=float)
    if data.ndim == 1:
        data = data.reshape(-1, 1)

    if np.isnan(data).any():
        data = data[~np.isnan(data).any(axis=1)]

    if data.size == 0:
        return np.empty((0, 0))

    covariance_matrix = np.cov(data, rowvar=False, ddof=1)
    return covariance_matrix

In [8]:
covariance_matrix = calculate_covariance_matrix(returns)

In [9]:
covariance_matrix

,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
A,0.000249,0.000057,0.000036,0.000110,0.000086,0.000008,0.000095,0.000053,0.000145,4.867537e-05,...,0.000034,0.000080,0.000012,0.000010,0.000069,0.000166,0.000053,0.000072,0.000102,0.000114
AAPL,0.000057,0.000167,0.000009,0.000078,0.000028,0.000020,0.000022,0.000030,0.000070,-4.398379e-07,...,0.000017,0.000090,-0.000024,-0.000004,0.000053,0.000023,0.000015,0.000026,0.000052,0.000062
ABBV,0.000036,0.000009,0.000269,-0.000005,0.000052,0.000024,0.000025,0.000045,-0.000004,1.574362e-05,...,0.000001,-0.000082,0.000031,-0.000025,0.000018,0.000025,0.000048,0.000003,-0.000041,-0.000017
ABNB,0.000110,0.000078,-0.000005,0.000425,0.000058,0.000016,0.000221,0.000220,0.000157,2.064221e-05,...,-0.000009,0.000181,-0.000025,-0.000070,0.000103,0.000267,-0.000011,0.000050,0.000194,0.000188
ABT,0.000086,0.000028,0.000052,0.000058,0.000241,0.000017,0.000035,0.000010,0.000032,5.715615e-05,...,0.000052,0.000018,0.000046,0.000023,0.000023,-0.000015,0.000069,0.000086,-0.000029,0.000086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XYZ,0.000166,0.000023,0.000025,0.000267,-0.000015,-0.000042,0.000355,0.000238,0.000173,-4.019185e-05,...,-0.000014,0.000152,-0.000084,-0.000028,0.000086,0.001185,0.000002,-0.000016,0.000090,0.000190
YUM,0.000053,0.000015,0.000048,-0.000011,0.000069,0.000065,0.000008,0.000019,0.000028,7.147239e-05,...,0.000081,0.000020,0.000035,0.000027,0.000052,0.000002,0.000217,0.000061,0.000034,-0.000035
ZBH,0.000072,0.000026,0.000003,0.000050,0.000086,0.000058,0.000021,0.000034,0.000053,1.477192e-04,...,0.000058,-0.000017,0.000052,0.000040,0.000021,-0.000016,0.000061,0.000450,0.000047,0.000108
ZBRA,0.000102,0.000052,-0.000041,0.000194,-0.000029,0.000017,0.000182,0.000163,0.000199,5.394096e-05,...,0.000088,0.000212,-0.000011,-0.000033,0.000122,0.000090,0.000034,0.000047,0.000736,0.000171
